# starling — DFXM analysis walkthrough

Step-by-step interactive workflow for **strain sweep**, **mosaicity**, and **3D strain-mosa** scans.
Device is auto-detected: CUDA on ESRF GPU nodes → MPS on Apple Silicon → CPU fallback.

**How to use:** Edit the `CONFIG` cell (Section 1), then run top to bottom with *Run All*.

The fitting step is a single `dset.analyze(mask=grain)` call — dispatch is automatic by
motor-dimension count (1 motor → `Gauss1DResult`, ≥2 motors → `GaussNDResult`), so 1-D, 2-D and
3-D scans all "just work" and the 3-D scan produces **fitted** maps (centre / width / mosaicity),
not just moments. Results are named objects (`res.mu`, `res.fwhm`, `res.mosaicity()`,
`res.orientation()`) with a `.raw` array for back-compat.


## 0 · Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
import time, os, h5py

import starling
from starling import properties

FWHM = 2 * np.sqrt(2 * np.log(2))   # sigma → FWHM conversion factor

print(f'starling {starling.__version__}  |  device: {starling.get_device(None)}')


def list_scans(dataset_folder):
    """Print every scan command (scan_id ending in .1) found in the dataset folder."""
    folder = Path(dataset_folder)
    h5_files = sorted(folder.glob('*.h5'))
    if not h5_files:
        print(f'No .h5 file found in {folder}'); return
    h5 = h5_files[0]
    print(f'File: {h5.name}\n')
    with h5py.File(h5) as f:
        keys = sorted(
            [k for k in f.keys() if '.' in k and k[0].isdigit() and k.endswith('.1')],
            key=lambda s: int(s.split('.')[0])
        )
        print(f'{"scan_id":>8}  command')
        print('-' * 90)
        for sid in keys:
            try:
                cmd = f[sid]['title'][()].decode()
                # also show frame count
                def _find_data(name, obj):
                    if (isinstance(obj, h5py.Dataset)
                            and obj.ndim == 3 and obj.shape[1] > 100):
                        raise StopIteration(obj.shape[0])
                nf = '?'
                try: f[sid].visititems(_find_data)
                except StopIteration as e: nf = e.args[0]
                print(f'{sid:>8}  [{nf} frames]  {cmd}')
            except Exception:
                pass
    return h5


def make_symmetric_norm(arr, mask, n_sigma=3, fallback=40e-3):
    """Return (vmin, vmax) centred on median of arr[mask], ±n_sigma*std."""
    if mask.any():
        centre = np.nanmedian(arr[mask])
        spread = max(np.nanstd(arr[mask]), fallback)
        return centre - n_sigma * spread, centre + n_sigma * spread
    return np.nanmin(arr), np.nanmax(arr)

## 1 · Choose your scan  ← **EDIT THIS CELL**

Uncomment **one** option block (A, B, or C), set `RAW_DATA` for your system, then run.

In [ ]:
# ─── change this path for the ESRF cluster ────────────────────────────────────
RAW_DATA = '/Volumes/Elements/MA6278/RAW_DATA'

# ═══════════════════════════════════════════════════════════════════════════════
# OPTION A — Strain sweep  (fscan2d ccmth × mu)
# Each scan is one time-point in the charging series.  Edit scan_id to pick
# a different time-point ('2.1', '3.1', … '37.1').
# ═══════════════════════════════════════════════════════════════════════════════
DATASET_NAME = 'DFXM_insitu_repaired_cell_charging/DFXM_insitu_repaired_cell_charging_g1_initial_strain_sweeps_during_charge_good'
SCAN_ID      = '1.1'
SCAN_TYPE    = 'strain_sweep'

# ═══════════════════════════════════════════════════════════════════════════════
# OPTION B — Mosaicity scan  (fscan2d chi × mu)
# ═══════════════════════════════════════════════════════════════════════════════
# DATASET_NAME = 'DFXM_insitu_repaired_cell_charging/DFXM_insitu_repaired_cell_charging_g1_mosa_projection_charging_1'
# SCAN_ID      = '1.1'
# SCAN_TYPE    = 'mosa'

# ═══════════════════════════════════════════════════════════════════════════════
# OPTION C — 3D strain-mosa layer  (chi × mu × obpitch stack)
# Load all 11 obpitch steps as a single 5-D dataset.
# ═══════════════════════════════════════════════════════════════════════════════
# DATASET_NAME = 'DFXM_insitu_repaired_cell_charging/DFXM_insitu_repaired_cell_charging_g1_2x_strain_mosa_layer_charging_2'
# SCAN_ID      = [f'{i}.1' for i in range(1, 12)]   # list → stacked load
# SCAN_TYPE    = 'strain_mosa_3d'

# ─── load-time ROI (optional but strongly recommended for large 3D scans) ─────
# Set to (row_min, row_max, col_min, col_max) to crop during loading — saves
# load time and RAM proportionally to ROI area.  A 512×512 crop on a 2048×2048
# detector cuts load time and RAM by ~16×.
# Use cell 2b (roi_picker) to find the correct values, then paste them here
# before running cell 3.
# Example: LOAD_ROI = (700, 1400, 850, 1550)
LOAD_ROI = None

# ─── preprocessing settings (rarely need changing) ───────────────────────────
SIG_THRESHOLD = 50      # counts: absolute floor, used only for fit-quality gating
N_BG_FRAMES   = 5       # darkest frames combined for BG_MODE 'lowest'/'mean'/'median'
BG_MODE       = 'lowest'   # background estimator:
                           #   'lowest'/'mean' — mean of N_BG_FRAMES darkest frames.
                           #     Best for the normal gradient grain (each pixel peaks
                           #     at a different motor step) — retains more grain than
                           #     darfix's median-over-all. KEEP THIS DEFAULT.
                           #   'percentile' — per-pixel low percentile over all frames.
                           #     Use for BROAD / always-lit grains (no clean dark frame),
                           #     i.e. when the retained-% diagnostic below is well < 90%.
                           #   'pmedian' — per-pixel median over all frames (darfix-style;
                           #     can eat a broad grain). 'median' — median of darkest frames.
BG_PERCENTILE = 10      # per-pixel percentile for BG_MODE='percentile' (10–25)
HOT_SIGMA     = 5.0     # hot-pixel detection: σ above local 3×3 median (robust MAD)
HOT_ONE_SIDED = True    # remove only bright spikes — never fill genuine interior dark features
HOT_MIN_SIGMA = 1.0     # floor on robust σ so mostly-zero (floored) frames can't flag grain pixels
ROI_THRESHOLD = 0.05    # auto_roi / grain_mask: fraction of z-sum max defining the grain
ROI_PAD       = 20      # pixels of padding around the grain box
CMAP_MDEG     = 40      # ±N mdeg symmetric range for orientation/strain maps

# ─── derived (do not edit) ────────────────────────────────────────────────────
_ds_base = Path(DATASET_NAME).name
H5_PATH  = f'{RAW_DATA}/{DATASET_NAME}/{_ds_base}.h5'
SCAN_MOTOR_3D = 'instrument/positioners/obpitch'

print(f'Dataset : {_ds_base}')
print(f'Scan ID : {SCAN_ID}')
print(f'Type    : {SCAN_TYPE}')
print(f'H5      : {H5_PATH}')
print(f'Exists  : {Path(H5_PATH).exists()}')
if LOAD_ROI:
    r1, r2, c1, c2 = LOAD_ROI
    print(f'Load ROI: rows {r1}–{r2}, cols {c1}–{c2}  ({r2-r1}×{c2-c1} px)')

In [ ]:
# -- map colour config (edit freely) ------------------------------------
# Colours for the result maps in Section 9 live in one shared dict so every
# map is consistent. Rebind an entry here to restyle all maps of that kind.
from starling.properties import DEFAULT_CMAPS
DEFAULT_CMAPS["mosaicity"] = "magma"   # darfix parity; set "plasma" for the old look
print("map cmaps:", DEFAULT_CMAPS)

## 2 · Browse available scans *(optional)*

Run this cell to see every scan command in your dataset — useful for picking a different `SCAN_ID` in the CONFIG cell.

## 2b · Lazy ROI picker *(run before loading — especially for large 3D scans)*

Streams a z-sum preview from a small strided subset of frames per scan — no full dataset load needed.
Drag a box on the detector image **or** type pixel coordinates, slide through scans to check the grain
is captured, then press **Confirm ROI**.  Copy the printed `LOAD_ROI` tuple into the CONFIG cell above
and re-run from cell 3 — only the selected pixels will be loaded.

In [ ]:
%matplotlib widget
starling.viz.roi_picker(H5_PATH, SCAN_ID)
# (switch back with `%matplotlib inline` before the plotting cells below)

In [ ]:
list_scans(f'{RAW_DATA}/{DATASET_NAME}')

## 3 · Load scan

Data is read from the BLISS HDF5 master file into RAM.  
Typical times: ~25 s from USB3 drive; much faster from NVMe or network-attached ESRF storage.  
For the 3D option, a progress bar shows the 11-scan stack being assembled.

In [ ]:
t0 = time.perf_counter()

if SCAN_TYPE == 'strain_mosa_3d':
    dset = starling.DataSet(H5_PATH, scan_id=SCAN_ID, scan_motor=SCAN_MOTOR_3D, roi=LOAD_ROI)
else:
    dset = starling.DataSet(H5_PATH, scan_id=SCAN_ID, roi=LOAD_ROI)

print(f'Loaded in {time.perf_counter()-t0:.1f} s')
print()
dset.info()
print()
print(f'data shape  : {dset.data.shape}')
print(f'dtype       : {dset.dtype}')
print(f'device      : {dset.device}')
sz_mb = dset.data.nbytes / 1e6
print(f'RAM in use  : {sz_mb:.0f} MB')

In [ ]:
# effective sample-plane pixel size, derived from the scan's obx/mainx/obpitch
# thin-lens geometry (falls back to the 6.5 um sensor pitch with a warning).
pixel_um = dset.pixel_size_um          # effective sample-plane pixel (motor-derived)
pixel_mm = pixel_um * 1e-3
print(f"effective pixel: {pixel_um*1000:.0f} nm  (nominal sensor pitch 6.5 um; M = {6.5/pixel_um:.1f}x)")

# manual override — uncomment to force a value (e.g. geometry motors missing):
# pixel_um = 6.5; pixel_mm = pixel_um * 1e-3

## 3b · Interactive denoise preview *(optional)*

Tune background / hot-pixel / ROI **live** and press **Apply** when happy. It is non-destructive — `dset.data` is untouched until Apply, and re-opening the widget always previews from the original data. If you use this, you can skip the manual preprocessing cells (4–6) below.


In [ ]:
%matplotlib widget
starling.viz.denoise_widget(dset, bg_n=N_BG_FRAMES, bg_mode=BG_MODE,
                            bg_percentile=BG_PERCENTILE, hot_sigma=HOT_SIGMA,
                            roi_threshold=ROI_THRESHOLD)
# The figure title shows the in-grain "signal retained %" and the third panel
# shows where counts were removed inside the grain — watch both while tuning.
# (switch back with `%matplotlib inline` before the plotting cells below)


## 4 · Background subtraction

`BG_MODE` (set in the CONFIG cell) chooses the estimator:

- **`lowest` / `mean`** *(default)* — pixel-wise mean of the `N_BG_FRAMES` globally-darkest frames. For the normal gradient grain those frames are off-Bragg *at the grain*, so the grain is **not** subtracted from itself. Empirically this retains more grain than darfix's median-over-all and is the right default.
- **`percentile`** — per-pixel low percentile (`BG_PERCENTILE`) over **all** frames. Use for **broad / always-lit grains**, where no frame is grain-free.
- **`pmedian`** — per-pixel median over all frames (darfix-style; can eat a broad grain). **`median`** — median of the darkest frames.

The cell prints the **in-grain signal-retained %** (processed integral vs the grain-safe percentile floor — *not* vs the raw integral, which legitimately includes the pedestal) and shows **where counts were removed inside the grain**, so you can confirm the background is not eating grain. If retained is well below 90 %, switch `BG_MODE` to `'percentile'` (or supply dark frames). Subtraction is clamped so `uint16` never wraps.

In [ ]:
bg = dset.estimate_background(n_lowest=N_BG_FRAMES, mode=BG_MODE,
                              percentile=BG_PERCENTILE)
_pct = f' (pctile={BG_PERCENTILE})' if BG_MODE == 'percentile' else ''
print(f'Background — mode={BG_MODE}{_pct}   mean: {bg.mean():.1f} cts   '
      f'max: {int(bg.max())} cts')

# ── diagnostic: does this background eat any in-grain signal? (run pre-subtract) ──
# 'retained' = grain signal kept in the LIT (near-Bragg) frames vs a per-pixel
# pedestal floor — pedestal removal and noise are excluded, so ~100% = no eating.
# 'overshoot' = how many counts bg sits above each grain pixel's own pedestal.
grain_fp = starling.preprocess.grain_mask(dset.data, threshold_rel=ROI_THRESHOLD)
diag = starling.preprocess.grain_signal_retained(dset.data, bg, mask=grain_fp)
signed = starling.preprocess.signed_zsum(dset.data, bg)   # pedestal-free integral
print(f'In-grain signal retained : {100*diag["retained"]:.1f}%   '
      f'(bg sits {diag["overshoot_median"]:+.1f} cts above the grain pedestal; '
      f'{diag["floored_px"]:,}/{diag["grain_px"]:,} px floored)')
print(f'Pedestal-free integral   : {signed[grain_fp].sum():.3e} cts  '
      f'(clean baseline for comparing time-points across the charging series)')
if diag["retained"] < 0.80:
    print('  ⚠ background is biased high at grain pixels — it is eating grain signal. '
          'Switch BG_MODE="percentile" (or supply dark frames) and re-check.')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
im = axes[0].imshow(bg.astype(float), cmap='hot')
axes[0].set_title(f'Background image ({BG_MODE})')
plt.colorbar(im, ax=axes[0], label='counts')
removed = np.where(grain_fp, diag['diff_map'], np.nan)
im = axes[1].imshow(removed, cmap='magma')
axes[1].set_title('Counts removed inside grain\n(flat = pedestal only; structured = eaten signal)')
plt.colorbar(im, ax=axes[1], label='counts')
plt.tight_layout()
plt.show()

dset.subtract(bg)
print('Background subtracted.')

## 5 · ROI selection — preview, then commit

Two cells: the **preview** proposes a box (cyan rectangle on the z-sum) without touching the data, and the **commit** cell applies whichever ROI you chose. Manual is the primary path — set `ROI_MANUAL = (row_min, row_max, col_min, col_max)` in the preview cell to override the auto suggestion (or use the lazy `roi_picker` widget in cell 2b).

**Auto-ROI** thresholds at `ROI_THRESHOLD × max(z-sum)` and adds `ROI_PAD` pixels of padding. Committing via `dset.auto_roi(apply=True)` also records the crop box so the RAW re-read (`save_raw_crop`) in the save cell can reconstruct the absolute detector ROI.

In [ ]:
# ── PREVIEW ROI — suggested box only, does NOT crop yet ──────────────────────
# Manual-first workflow: the lazy roi_picker widget (cell 2b) is the primary way
# to choose a box. This auto box is a fallback suggestion — accept it, or set
# ROI_MANUAL below and re-run. The crop itself happens in the next cell.
zsum_full = dset.data.reshape(*dset.data.shape[:2], -1).sum(-1).astype(float)
ny_full, nx_full = zsum_full.shape

roi = dset.auto_roi(threshold_rel=ROI_THRESHOLD, pad=ROI_PAD, apply=False)
r1, r2, c1, c2 = roi
print(f'Suggested ROI : rows {r1}–{r2}, cols {c1}–{c2}')
print(f'                {r2-r1} × {c2-c1} px = '
      f'{(r2-r1)*pixel_mm:.2f} × {(c2-c1)*pixel_mm:.2f} mm')

# ── Manual override — set to (row_min, row_max, col_min, col_max) or leave None ─
ROI_MANUAL = None
# ROI_MANUAL = (800, 1300, 900, 1500)

fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(zsum_full, cmap='hot', interpolation='nearest')
rect = plt.Rectangle((c1, r1), c2-c1, r2-r1, edgecolor='cyan', facecolor='none', lw=2)
ax.add_patch(rect)
ax.set_title('suggested box — edit ROI below or accept')
ax.set_xlabel('detector column'); ax.set_ylabel('detector row')
plt.tight_layout()
plt.show()

In [ ]:
# ── COMMIT ROI — apply whichever box was chosen (manual wins) ────────────────
if ROI_MANUAL is not None:
    r1, r2, c1, c2 = ROI_MANUAL
    dset.data = np.ascontiguousarray(dset.data[r1:r2, c1:c2])
    print(f'Using manual ROI: {ROI_MANUAL}')
else:
    # dset.auto_roi(apply=True) crops in place AND records the box so
    # dset.final_roi() can reconstruct the absolute detector ROI later
    # (needed for save_raw_crop in the save cell).
    r1, r2, c1, c2 = dset.auto_roi(threshold_rel=ROI_THRESHOLD, pad=ROI_PAD, apply=True)
    print(f'Applied auto ROI: rows {r1}–{r2}, cols {c1}–{c2}')

print(f'Data shape after crop: {dset.data.shape}')

## 6 · Hot-pixel removal *(optional)*

Replaces isolated hot pixels with their 3×3 spatial median. Detection threshold: `HOT_SIGMA × MAD` per frame (robust). Grain-safe settings from the CONFIG cell:

- **`HOT_ONE_SIDED=True`** — removes only *bright* spikes, so a genuine interior dark feature (void / weaker sub-grain) is never filled brighter. Real zingers, even on the grain, are still removed.
- **`HOT_MIN_SIGMA=1.0`** — floors the robust σ so a mostly-zero (post-subtraction, floored) frame can't collapse the threshold and flag genuine grain-interior pixels.

Runs batched on the compute device (CUDA/MPS; ~0.05 s/frame on MPS vs ~0.4 s/frame for the old scipy loop — pass `backend='scipy'` to `starling.preprocess.remove_hot_pixels` for the reference path). Skip this cell if you do not see obvious streaks in the grain maps. *(To freeze the grain entirely — never touch any grain pixel, accepting that on-grain zingers survive — pass `protect=starling.preprocess.grain_mask(dset.data, threshold_rel=ROI_THRESHOLD)`.)*

In [ ]:
n_frames = int(np.prod(dset.data.shape[2:]))
ny, nx = dset.data.shape[:2]
print(f'Running hot-pixel filter on {n_frames} frames of {ny}×{nx} px '
      f'— torch-batched, expect ~{n_frames*0.05:.0f} s on MPS ...')
t0 = time.perf_counter()
dset.remove_hot_pixels(n_sigma=HOT_SIGMA, one_sided=HOT_ONE_SIDED,
                       min_sigma=HOT_MIN_SIGMA)
print(f'Done in {time.perf_counter()-t0:.1f} s  '
      f'(one-sided={HOT_ONE_SIDED}, min_sigma={HOT_MIN_SIGMA})')

In [ ]:
# optional darfix-parity hard threshold (usually unnecessary — clamped
# background subtraction already removes the pedestal):
# dset.threshold_removal(bottom=..., top=...)

## 7 · Grain overview

Z-sum after all preprocessing.  The signal mask (pixels above `SIG_THRESHOLD` in any frame) is used throughout to exclude background in result maps and statistics.

In [ ]:
ny, nx = dset.data.shape[:2]
zsum = dset.data.reshape(ny, nx, -1).sum(-1).astype(float)

# ONE grain footprint, used for BOTH fitting and display: the integrated z-sum
# thresholded at ROI_THRESHOLD*max, morphologically closed and hole-FILLED. Using
# the same filled mask everywhere means interior pixels the fit computed are never
# shown as spurious "holes". (Replaces the old per-frame max>SIG_THRESHOLD mask,
# which was unfilled and made fitted grain-interior pixels look like lost signal.)
grain = starling.preprocess.grain_mask(dset.data, threshold_rel=ROI_THRESHOLD)
sig_mask = grain    # alias: every result map / statistic below uses this footprint

# informational only — where a single frame exceeds the absolute SIG_THRESHOLD
sig_any = dset.data.reshape(ny, nx, -1).max(-1) > SIG_THRESHOLD

extent = [0, nx * pixel_mm, ny * pixel_mm, 0]   # mm

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
im = ax.imshow(zsum, cmap='hot', extent=extent, interpolation='nearest')
ax.set_xlabel('mm'); ax.set_ylabel('mm')
ax.set_title('Grain — z-sum (integrated counts)')
plt.colorbar(im, ax=ax, label='counts')

ax = axes[1]
ax.imshow(grain.astype(float), cmap='Greys_r', extent=extent,
          vmin=0, vmax=1, interpolation='nearest')
ax.set_xlabel('mm'); ax.set_ylabel('mm')
ax.set_title('Grain footprint (filled) — mask used for fits & maps')

plt.tight_layout()
plt.show()

print(f'Grain pixels        : {grain.sum():,} / {ny*nx:,}  '
      f'({100*grain.mean():.1f} % of ROI)')
print(f'…of which >{SIG_THRESHOLD} cts in a single frame: {(grain & sig_any).sum():,} '
      f'(low-signal interior pixels are kept by the filled mask)')

## 8 · Fitting

The correct fit is selected automatically from `SCAN_TYPE`:

| Type | Method | Output |
|------|--------|--------|
| `strain_sweep` | `fit_1D_gaussian` per µ layer | (ny, nx, 6, n_µ) — [A, σ, ccmth_centre, k, m, ok] |
| `mosa` | `moments` + `fit_2D_gaussian` | moments (ny,nx,2/2×2); gauss (ny,nx,8) |
| `strain_mosa_3d` | `moments` (3D) | (ny, nx, 3) + (ny, nx, 3, 3) |

> **Note:** the first `fit_1D_gaussian` / `fit_2D_gaussian` call per session pays a ~5–10 s `torch.compile` warm-up.  Subsequent calls use the cached kernel and are fast.

In [ ]:
# ── Grain mask: reuse the SAME filled footprint built in Section 7 so the fits and
#    the displayed maps cover identical pixels (faster; identical on-grain values) ──
grain = globals().get('grain')
if grain is None or grain.shape != dset.data.shape[:2]:
    grain = starling.preprocess.grain_mask(dset.data, threshold_rel=ROI_THRESHOLD)
print(f'grain mask: {grain.sum():,} / {grain.size:,} px ({100*grain.mean():.1f}%)')

from starling.properties import Gauss1DResult

if SCAN_TYPE == 'strain_sweep':
    # Depth-resolved strain: each µ layer is an independent 1-D ccmth rocking
    # scan, so fit per layer. (A single rocking scan would just be
    # `dset.analyze()` -> Gauss1DResult directly.)
    n_ccmth, n_mu = dset.data.shape[2], dset.data.shape[3]
    mu_values = dset.motors[1, 0, :]
    print(f'Strain sweep: {n_ccmth} ccmth × {n_mu} µ layers '
          f'(compile warm-up on first layer ~5-10 s)...')
    res = []
    for k in range(n_mu):
        t0 = time.perf_counter()
        p = properties.fit_1D_gaussian(dset.data[:, :, :, k], dset.motors[:1, :, k],
                                       mask=grain, device=dset.device)
        r = Gauss1DResult.from_raw(p)
        ok_k = (r.success > 0.5) & grain & (r.A > SIG_THRESHOLD)
        print(f'  µ layer {k:2d}  µ={mu_values[k]:.3f}°  success: {ok_k.sum():5,}  '
              f'({time.perf_counter()-t0:.2f} s)')
        res.append(r)

else:
    # Mosa (2 motors) and 3D strain-mosa (3 motors) dispatch automatically:
    # analyze() returns a GaussNDResult — the 3D scan produces FITTED maps,
    # not just moments.
    print(f'analyze(): {dset.n_motor_dims} motor dims '
          f'(compile warm-up on first call ~5-10 s)...')
    t0 = time.perf_counter()
    res = dset.analyze(mask=grain)
    print(f'  -> {type(res).__name__}  D={getattr(res, "D", "-")}  '
          f'success {100*(res.success > 0).mean():.1f}%  ({time.perf_counter()-t0:.2f} s)')


### Fit-status convention

Every result map below is coloured by a per-pixel **fit status** so you can tell *why* a pixel is blank:

- **light grey** — no grain signal here (outside the grain mask).
- **orange outline** — the peak was **clipped by the scan range** (the grain is still there — the scan window was too narrow to capture the whole rocking curve). For the centre-of-mass maps the pixel still shows a *display-only* boundary estimate (`clamp_edge_estimate`); never use those clamped values quantitatively.
- **dark grey** — the fit **failed** (Gauss-Newton did not converge to a valid peak).

`status_legend(...)` on the first map of each figure spells this out on the plot itself.

In [ ]:
# Per-pixel fit status: 0 no-signal / 1 ok / 2 edge-clipped / 3 failed.
from starling.properties import clamp_edge_estimate, motor_ranges_steps

if SCAN_TYPE == 'strain_sweep':
    # res is a list of per-µ-layer 1-D results; status is built per layer in the
    # map cell below (each layer has its own 1-D motor axis).
    status = None
    mu_display = None
else:
    status = dset.fit_status(res, mask=grain)      # 0 no-signal / 1 ok / 2 edge-clipped / 3 failed
    ranges, steps = motor_ranges_steps(dset.motors)
    mu_display = clamp_edge_estimate(res.mu, ranges)   # display-only values for edge pixels
    n = status.size
    print(f'fit status — ok {100*(status==1).sum()/n:.1f}%   '
          f'edge-clipped {100*(status==2).sum()/n:.1f}%   '
          f'failed {100*(status==3).sum()/n:.1f}%   '
          f'(no-signal {100*(status==0).sum()/n:.1f}%)')

## 9 · Result maps

All maps are routed through the shared `starling.properties.imshow_map` helper, so they share one look: off-grain pixels render **light grey**, edge-clipped regions are **outlined in orange** (with a display-only boundary estimate on the centre maps), failed fits are **dark grey**, and every panel gets a colorbar. Colours come from the `DEFAULT_CMAPS` config cell near the top.

Orientation / strain centre maps are shown as **deviations from the grain median** in **millidegrees**; the orientation colour stamp is the darfix-style fixed-range square (mean orientation, not mosaicity).

In [ ]:
from starling.properties import imshow_map, status_legend, robust_limits


def dev_mdeg(arr, m):
    """Deviation from the in-mask median, in millidegrees."""
    return (arr - np.nanmedian(arr[m])) * 1000


# ─── STRAIN SWEEP — per-µ-layer 1D fits (Gauss1DResult list) ─────────────────
if SCAN_TYPE == 'strain_sweep':
    n_mu = len(res)
    n_show = min(n_mu, 6)
    k_show = np.round(np.linspace(0, n_mu - 1, n_show)).astype(int)
    fig, axes = plt.subplots(3, n_show, figsize=(3.5 * n_show, 9), constrained_layout=True)
    if n_show == 1:
        axes = axes[:, None]
    for col, k in enumerate(k_show):
        r = res[k]
        ok = (r.success > 0.5) & sig_mask & (r.A > SIG_THRESHOLD)
        # 1-D layer status: ok / failed (on-grain, bad fit) / no-signal (off-grain)
        status_k = np.where(ok, 1, np.where(grain, 3, 0)).astype(np.int8)
        imshow_map(axes[0, col], dev_mdeg(r.mu, ok), fit_status=status_k, cmap='com',
                   vmin=-CMAP_MDEG, vmax=CMAP_MDEG, interpolation='nearest',
                   cbar_label='mdeg' if col == n_show - 1 else None)
        axes[0, col].set_title(f'µ={mu_values[k]:.3f}°', fontsize=9)
        if col == 0:
            axes[0, col].set_ylabel('Δccmth (mdeg)')
            status_legend(axes[0, col], status_k)
        imshow_map(axes[1, col], r.fwhm, fit_status=status_k, cmap='fwhm',
                   vmin=0, vmax=0.02, interpolation='nearest',
                   cbar_label='°' if col == n_show - 1 else None)
        if col == 0:
            axes[1, col].set_ylabel('FWHM (°)')
        imshow_map(axes[2, col], r.A, fit_status=status_k, cmap='amplitude', vmin=0,
                   interpolation='nearest', cbar_label='cts' if col == n_show - 1 else None)
        if col == 0:
            axes[2, col].set_ylabel('Amplitude (cts)')
    for row in axes:
        for ax in row:
            ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(f'Strain sweep maps — {_ds_base}', y=1.01)
    plt.show()

    med_centre, med_fwhm = [], []
    for r in res:
        ok = (r.success > 0.5) & sig_mask & (r.A > SIG_THRESHOLD)
        med_centre.append(np.nanmedian(r.mu[ok]) if ok.any() else np.nan)
        med_fwhm.append(np.nanmedian(r.fwhm[ok]) if ok.any() else np.nan)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(mu_values, med_centre, 'o-', color='steelblue', lw=2, ms=6)
    axes[0].set_xlabel('µ (°)'); axes[0].set_ylabel('Median ccmth peak centre (°)')
    axes[0].set_title('Strain depth profile'); axes[0].grid(True, alpha=0.3)
    axes[1].plot(mu_values, med_fwhm, 's-', color='coral', lw=2, ms=6)
    axes[1].set_xlabel('µ (°)'); axes[1].set_ylabel('Median FWHM (°)')
    axes[1].set_title('Rocking width vs depth'); axes[1].grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()


# ─── MOSA — 2D Gaussian (GaussNDResult, D=2) ─────────────────────────────────
elif SCAN_TYPE == 'mosa':
    ok = (res.success > 0.5) & sig_mask
    fig, axes = plt.subplots(2, 3, figsize=(15, 9), constrained_layout=True)
    ax = axes.flat
    # centre = ORIENTATION (first moment); FWHM/spread = MOSAICITY (second moment)
    imshow_map(ax[0], dev_mdeg(res.mu[..., 0], ok), fit_status=status, cmap='com',
               vmin=-CMAP_MDEG, vmax=CMAP_MDEG, cbar_label='mdeg',
               show_clamped=dev_mdeg(mu_display[..., 0], ok), interpolation='nearest')
    ax[0].set_title('Δchi orientation (mdeg)'); status_legend(ax[0], status)
    imshow_map(ax[1], dev_mdeg(res.mu[..., 1], ok), fit_status=status, cmap='com',
               vmin=-CMAP_MDEG, vmax=CMAP_MDEG, cbar_label='mdeg',
               show_clamped=dev_mdeg(mu_display[..., 1], ok), interpolation='nearest')
    ax[1].set_title('Δµ orientation (mdeg)')
    imshow_map(ax[2], res.A, fit_status=status, cmap='amplitude', vmin=0,
               cbar_label='cts', interpolation='nearest')
    ax[2].set_title('Amplitude (cts)')
    imshow_map(ax[3], res.fwhm[..., 0] * 1000, fit_status=status, cmap='fwhm', vmin=0,
               cbar_label='mdeg', interpolation='nearest')
    ax[3].set_title('FWHM chi (mdeg)')
    imshow_map(ax[4], res.fwhm[..., 1] * 1000, fit_status=status, cmap='fwhm', vmin=0,
               cbar_label='mdeg', interpolation='nearest')
    ax[4].set_title('FWHM µ (mdeg)')
    imshow_map(ax[5], res.mosaicity(mode='scalar') * FWHM * 1000, fit_status=status,
               cmap='mosaicity', vmin=0, cbar_label='mdeg FWHM', interpolation='nearest')
    ax[5].set_title('Mosaicity = orientation spread (mdeg FWHM)')
    for a in ax:
        a.set_xticks([]); a.set_yticks([])
    fig.suptitle(f'Mosaicity maps — {_ds_base}  (centre = orientation, spread = mosaicity)')
    plt.show()

    # orientation — darfix-style fixed-range square colour stamp (mean orientation)
    rgb, key, vrange = res.orientation_stamp(mask=ok)
    fig, (ax0, ax1) = plt.subplots(1, 2, width_ratios=(4, 1), figsize=(11, 4))
    ax0.imshow(rgb); ax0.set_title('orientation (darfix-style colour stamp)')
    ax0.set_xticks([]); ax0.set_yticks([])
    (lo0, hi0), (lo1, hi1) = vrange
    ax1.imshow(key, origin='lower', extent=(lo0, hi0, lo1, hi1), aspect='auto')
    ax1.set_xlabel('chi COM (deg)'); ax1.set_ylabel('mu COM (deg)'); ax1.set_title('colour key')
    plt.tight_layout(); plt.show()


# ─── 3D STRAIN-MOSA — FITTED maps (GaussNDResult, D=3) ───────────────────────
elif SCAN_TYPE == 'strain_mosa_3d':
    ob_values = dset.motors[2, 0, 0, :]
    n_ob = dset.data.shape[4]
    ok = (res.success > 0.5) & sig_mask

    # intensity depth profile (where the grain sits in obpitch)
    I_per_layer = dset.data.sum(axis=(0, 1, 2, 3))
    peak_layer = int(np.argmax(I_per_layer))
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(ob_values, I_per_layer / I_per_layer.max(), 'o-', color='steelblue', lw=2, ms=7)
    ax.axvline(ob_values[peak_layer], ls='--', color='crimson', lw=1.5,
               label=f'peak: {ob_values[peak_layer]:.4f}° (layer {peak_layer+1})')
    ax.set_xlabel('obpitch (°)'); ax.set_ylabel('Normalised integrated intensity')
    ax.set_title('Depth profile — intensity vs obpitch layer'); ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

    # fitted centre maps (chi, µ, obpitch) — from the 3D Gaussian fit
    labels = ['chi centre', 'µ centre', 'obpitch centre']
    cmaps = ['com', 'com', 'strain']
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)
    for i in range(3):
        vlo, vhi = robust_limits(res.mu[..., i], ok, symmetric=False)
        imshow_map(axes[i], res.mu[..., i], fit_status=status, cmap=cmaps[i],
                   vmin=vlo, vmax=vhi, cbar_label='°', show_clamped=mu_display[..., i],
                   interpolation='nearest')
        axes[i].set_title(labels[i]); axes[i].set_xticks([]); axes[i].set_yticks([])
        if i == 0:
            status_legend(axes[i], status)
    fig.suptitle(f'3D fitted centre maps — {_ds_base}')
    plt.show()

    # mosaicity from the chi,µ orientation block only (excludes obpitch/strain axis)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
    imshow_map(axes[0], res.mosaicity(axes=(0, 1), mode='scalar') * FWHM * 1000,
               fit_status=status, cmap='mosaicity', cbar_label='mdeg FWHM',
               interpolation='nearest')
    axes[0].set_title('Mosaicity (chi,µ spread, mdeg FWHM)')
    axes[0].set_xticks([]); axes[0].set_yticks([])
    imshow_map(axes[1], res.A, fit_status=status, cmap='amplitude', cbar_label='cts',
               interpolation='nearest')
    axes[1].set_title('Amplitude (cts)')
    axes[1].set_xticks([]); axes[1].set_yticks([])
    fig.suptitle(f'3D mosaicity + amplitude — {_ds_base}')
    plt.show()

## 10 · Statistics

Distributions over all grain pixels (masked to `sig_mask`).  Each histogram shows the **spread within a single grain image** — the width reflects real orientation / strain heterogeneity inside the grain.

In [ ]:
print('\n' + '─' * 60)

if SCAN_TYPE == 'strain_sweep':
    best_k = max(range(len(res)),
                 key=lambda k: ((res[k].success > 0.5) & sig_mask
                                & (res[k].A > SIG_THRESHOLD)).sum())
    r = res[best_k]
    ok = (r.success > 0.5) & sig_mask & (r.A > SIG_THRESHOLD)
    centres_deg, fwhm_deg, amps = r.mu[ok], r.fwhm[ok], r.A[ok]
    print(f'Strain sweep — best µ layer: k={best_k}  µ={mu_values[best_k]:.3f}°')
    print(f'  Grain pixels with good fit : {ok.sum():,}')
    print(f'  ccmth peak centre          : {np.median(centres_deg):.5f}°  '
          f'±{np.std(centres_deg)*1000:.2f} mdeg spread')
    print(f'  Median FWHM                : {np.median(fwhm_deg)*1000:.2f} mdeg')
    print(f'  Median amplitude           : {np.median(amps):.0f} counts')

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].hist((centres_deg - np.median(centres_deg)) * 1000, bins=80,
                 color='steelblue', edgecolor='none', alpha=0.85)
    axes[0].axvline(0, color='k', lw=1.5, ls='--')
    axes[0].set_xlabel('Δccmth from median (mdeg)'); axes[0].set_ylabel('Grain pixels')
    axes[0].set_title(f'Strain distribution — µ={mu_values[best_k]:.3f}°'); axes[0].grid(True, alpha=0.2)
    axes[1].hist(fwhm_deg * 1000, bins=80, color='coral', edgecolor='none', alpha=0.85)
    axes[1].set_xlabel('FWHM (mdeg)'); axes[1].set_ylabel('Grain pixels')
    axes[1].set_title('Rocking width distribution'); axes[1].grid(True, alpha=0.2)
    axes[2].plot(mu_values,
                 [(res[k].A[sig_mask].mean() if sig_mask.any() else 0) for k in range(len(res))],
                 'o-', color='seagreen', lw=2)
    axes[2].set_xlabel('µ (°)'); axes[2].set_ylabel('Mean amplitude (cts)')
    axes[2].set_title('Signal strength vs depth'); axes[2].grid(True, alpha=0.2)
    plt.tight_layout(); plt.show()


elif SCAN_TYPE == 'mosa':
    ok = (res.success > 0.5) & sig_mask
    chi_px, mu_px = res.mu[..., 0][ok], res.mu[..., 1][ok]
    fchi_px, fmu_px = res.fwhm[..., 0][ok] * 1000, res.fwhm[..., 1][ok] * 1000
    mosa_px = res.mosaicity(mode='scalar')[ok] * FWHM * 1000
    print('Mosaicity statistics:')
    print(f'  Grain pixels with good fit : {ok.sum():,}')
    print(f'  chi orientation centre     : {np.median(chi_px):.4f}°  ±{np.std(chi_px)*1000:.2f} mdeg')
    print(f'  µ orientation centre       : {np.median(mu_px):.4f}°  ±{np.std(mu_px)*1000:.2f} mdeg')
    print(f'  Median FWHM chi            : {np.median(fchi_px):.1f} mdeg')
    print(f'  Median FWHM µ              : {np.median(fmu_px):.1f} mdeg')
    print(f'  Median total mosaicity     : {np.median(mosa_px):.1f} mdeg (orientation spread)')

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].hist((chi_px - np.median(chi_px)) * 1000, bins=80, color='steelblue', edgecolor='none', alpha=0.85)
    axes[0].axvline(0, color='k', lw=1.5, ls='--')
    axes[0].set_xlabel('Δchi orientation (mdeg)'); axes[0].set_ylabel('Grain pixels')
    axes[0].set_title('chi orientation spread'); axes[0].grid(True, alpha=0.2)
    axes[1].hist((mu_px - np.median(mu_px)) * 1000, bins=80, color='coral', edgecolor='none', alpha=0.85)
    axes[1].axvline(0, color='k', lw=1.5, ls='--')
    axes[1].set_xlabel('Δµ orientation (mdeg)'); axes[1].set_title('µ orientation spread'); axes[1].grid(True, alpha=0.2)
    axes[2].hist(fchi_px, bins=80, color='seagreen', edgecolor='none', alpha=0.7, label='chi')
    axes[2].hist(fmu_px, bins=80, color='purple', edgecolor='none', alpha=0.5, label='µ')
    axes[2].set_xlabel('FWHM (mdeg)'); axes[2].set_title('Rocking-curve width distributions')
    axes[2].legend(); axes[2].grid(True, alpha=0.2)
    plt.tight_layout(); plt.show()


elif SCAN_TYPE == 'strain_mosa_3d':
    ok = (res.success > 0.5) & sig_mask
    print('3D strain-mosa statistics (from the fitted 3D Gaussian):')
    print(f'  Grain pixels with good fit : {ok.sum():,}')
    for i, lbl in enumerate(['chi', 'µ', 'obpitch']):
        v = res.mu[..., i][ok]
        print(f'  {lbl} centre : {np.median(v):.4f}°  ±{np.std(v)*1000:.2f} mdeg spread')
    mos = res.mosaicity(axes=(0, 1), mode='scalar')[ok] * FWHM * 1000
    print(f'  chi,µ mosaicity (spread)   : {np.median(mos):.1f} mdeg FWHM')

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    colors_3d = ['steelblue', 'coral', 'seagreen']
    for i, lbl in enumerate(['chi centre', 'µ centre', 'obpitch centre']):
        v = res.mu[..., i][ok]
        axes[i].hist((v - np.median(v)) * 1000, bins=80, color=colors_3d[i], edgecolor='none', alpha=0.85)
        axes[i].axvline(0, color='k', lw=1.5, ls='--')
        axes[i].set_xlabel(f'Δ{lbl} (mdeg)'); axes[i].set_ylabel('Grain pixels')
        axes[i].set_title(f'{lbl} distribution'); axes[i].grid(True, alpha=0.2)
    plt.tight_layout(); plt.show()


## 11 · Save results

In [ ]:
OUT = f'/Users/matt/Lab/projects/DFXM/analysis/notebook_{SCAN_TYPE}'
os.makedirs(OUT, exist_ok=True)

# ── legacy per-array dumps (kept for back-compat) ────────────────────────────
np.save(f'{OUT}/sig_mask.npy', sig_mask)
np.save(f'{OUT}/grain_mask.npy', grain)
np.save(f'{OUT}/motors.npy', dset.motors)
if SCAN_TYPE == 'strain_sweep':
    np.save(f'{OUT}/params.npy', np.stack([r.raw for r in res], axis=-1))
else:
    res.to_h5(f'{OUT}/result.h5')

# ── self-contained analysis bundle ───────────────────────────────────────────
masks = {'sig_mask': sig_mask, 'grain': grain}
if status is not None:
    masks['fit_status'] = status
starling.save_bundle(
    f'{OUT}/bundle.h5', dset,
    results={'fit': res},   # list[Gauss1DResult] or GaussNDResult — both accepted
    masks=masks,
    save_raw_crop=True,     # also persists /data/raw_cube: ROI-cropped RAW data before
                            # bg-subtract/hot-pixel — one extra read pass
)

# ── NeXus export (single result objects only; skips the strain-sweep list) ───
if SCAN_TYPE != 'strain_sweep':
    dset.save_nexus(f'{OUT}/result.nxs', res, fit_status=status)

print(f'Saved to {OUT}/')
for f in sorted(os.listdir(OUT)):
    sz = os.path.getsize(f'{OUT}/{f}') / 1e6
    print(f'  {f:<35}  {sz:.1f} MB')

## 12 · Pixel inspector *(optional)*

Click any pixel on a map to see its **raw rocking curve(s)** with the fitted Gaussian overlaid — the fastest way to sanity-check why a pixel came out edge-clipped or failed. Uncomment to launch (needs the `widget` backend).

In [ ]:
# %matplotlib widget
# starling.viz.pixel_inspector(dset, result=res, fit_status=status)